# 19 · Stage 3 v5-D — fixed v5-C overlap acceleration validation

목적은 **v5-D를 바로 제출하지 않고**, v5-C에서 이미 선택된 validation protocol을
그대로 고정한 상태에서 acceleration 개선이 실제로 남는지 확인하는 것이다.

이 notebook은 다음 원칙을 지킨다.

1. `history.csv / initial_validation.json / summary.json`으로 E1/E2를 먼저 진단한다.
2. `best.pt / best_proxy.pt / best_accel_proxy.pt`의 epoch·SHA256을 확인한다.
3. v5-C의 **동일 25개 complete segment / 동일 tune-holdout split**을 재사용한다.
4. overlap은 v5-C 선택값을 그대로 사용한다: `T=32`, `stride=8`, `center_floor=0.25`.
5. v5-C production fusion도 그대로 사용한다. 특히 기존 accel ordinal fusion은
   선택 결과대로 `accel_weight=0`인 상태를 유지한다.
6. v5-D의 새 3-state head는 **threshold/temperature를 재튜닝하지 않고**
   단 하나의 fusion weight만 tune split에서 탐색한다.
7. holdout/full 및 `ACCELERATING`, `dynamic→CONSTANT` 안전성까지 통과해야
   local acceptance로 표시한다.
8. released 50 labels는 선택에 사용하지 않는다. 마지막 optional cell에서만
   sanity check로 사용할 수 있다.

출력:
- `checkpoint_fingerprints.csv`
- `overlap_checkpoint_comparison.csv`
- `state_fusion_grid.csv`
- `v5d_overlap_accel_report.json`

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import os
import shutil
import subprocess
import sys
import time
import zipfile
from pathlib import Path

from google.colab import drive

try:
    drive.mount("/content/drive", force_remount=False)
except Exception:
    drive.mount("/content/drive", force_remount=True)

REPO = Path("/content/Blackbox-Detection")
REPO_URL = "https://github.com/sangchun1/Blackbox-Detection.git"
BRANCH = "stage3-sangchun"

if not (REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "--depth", "1", "--branch", BRANCH,
         "--single-branch", REPO_URL, str(REPO)],
        check=True,
    )
else:
    current = subprocess.run(
        ["git", "-C", str(REPO), "branch", "--show-current"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if current != BRANCH:
        subprocess.run(["git", "-C", str(REPO), "checkout", BRANCH], check=True)

    dirty = subprocess.run(
        ["git", "-C", str(REPO), "status", "--porcelain"],
        check=True, capture_output=True, text=True,
    ).stdout.strip()
    if not dirty:
        subprocess.run(
            ["git", "-C", str(REPO), "pull", "--ff-only", "origin", BRANCH],
            check=True,
        )
    else:
        print("WARNING: local repo dirty; git pull skipped")

subprocess.run(
    [
        sys.executable, "-m", "pip", "install", "-q",
        "--upgrade-strategy", "only-if-needed",
        "timm==1.0.15",
        "fvcore==0.1.5.post20221221",
        "iopath==0.1.10",
        "yacs==0.1.8",
        "einops==0.8.1",
        "easydict==1.13",
    ],
    check=True,
)

SRC = REPO / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

import numpy as np
import pandas as pd
import torch
import yaml
from sklearn.metrics import f1_score

from blackbox_detection.stage3.metrics import (
    ACCEL_CLASSES,
    STEER_CLASSES,
    dacon_stage3_metrics,
)
from blackbox_detection.stage3.proxy_metrics import (
    ProxyRule,
    assert_dacon_metric_contract,
)
from blackbox_detection.stage3.schema import read_frame_table
from blackbox_detection.stage3.v5c_inference import (
    DecisionRule,
    FusionConfig,
    center_weights,
    decode_sampled_video,
    labels_with_aux_fusion,
    score_proxy_table,
    smooth_feature_table,
    threshold_tag,
    window_starts,
)
from blackbox_detection.utils import seed_everything
from blackbox_detection.utils.checkpoint import load_checkpoint

DRIVE_ROOT = Path("/content/drive/MyDrive/Blackbox-Detection")
DATA_ROOT = DRIVE_ROOT / "DATASET"
COMMA_ROOT = DATA_ROOT / "comma2k19" / "processed" / "v1"
MANIFEST_ROOT = DRIVE_ROOT / "manifests/stage3/v1"
OUTPUT_ROOT = DRIVE_ROOT / "outputs/stage3"
PRETRAINED_ROOT = DRIVE_ROOT / "pretrained"

LOCAL_PRETRAINED_ROOT = Path("/content/pretrained")
LOCAL_PRETRAINED_ROOT.mkdir(parents=True, exist_ok=True)

V5C_RUN_NAME = "vjepa21b_can_v5c_overlap_auxfusion"
V5D_RUN_NAME = "vjepa21b_can_v5d_accel_decision"
V5C_RUN = OUTPUT_ROOT / V5C_RUN_NAME
V5D_RUN = OUTPUT_ROOT / V5D_RUN_NAME
REPORT_DIR = V5D_RUN / "overlap_validation"
REPORT_DIR.mkdir(parents=True, exist_ok=True)

V5C_CFG_PATH = REPO / "configs/stage3/vjepa21b_can_v5c.yaml"
V5D_CFG_PATH = REPO / "configs/stage3/vjepa21b_can_v5d.yaml"
v5c_cfg = yaml.safe_load(V5C_CFG_PATH.read_text(encoding="utf-8"))
v5d_cfg = yaml.safe_load(V5D_CFG_PATH.read_text(encoding="utf-8"))

stats = json.loads(
    (MANIFEST_ROOT / "target_stats.json").read_text(encoding="utf-8")
)

SEED = int(v5d_cfg["seed"])
seed_everything(SEED, deterministic=False)
assert_dacon_metric_contract()

GIT_COMMIT = subprocess.run(
    ["git", "-C", str(REPO), "rev-parse", "HEAD"],
    check=True, capture_output=True, text=True,
).stdout.strip()

print("GPU        :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("repo HEAD  :", GIT_COMMIT)
print("v5-C run   :", V5C_RUN)
print("v5-D run   :", V5D_RUN)
print("report dir :", REPORT_DIR)
print("metric     : PASS")


Mounted at /content/drive
GPU        : NVIDIA L4
repo HEAD  : 1ec2ba2e210905040a889ea5cd780d9da4aea462
v5-C run   : /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5c_overlap_auxfusion
v5-D run   : /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5d_accel_decision
report dir : /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5d_accel_decision/overlap_validation
metric     : PASS


## 1. E1 / E2와 warm-start anchor 분석

`history.csv`, `initial_validation.json`, `summary.json`을 Drive의 실제 v5-D run에서 읽는다.
여기서는 overlap이 아니라 학습 당시 고정 800-window diagnostic의 변화를 확인한다.

특히 다음을 같이 본다.

- robust Stage3 / accel / steer
- rule별 accel macro-F1
- `ACCELERATING / DECELERATING / CONSTANT / STOPPED`
- `dynamic_to_constant_rate`
- acceleration correlation / std ratio / slope / p95
- regression MAE / RMSE
- 새 3-state CE와 기존 ordinal diagnostic

In [2]:
HISTORY_PATH = V5D_RUN / "history.csv"
INITIAL_PATH = V5D_RUN / "initial_validation.json"
SUMMARY_PATH = V5D_RUN / "summary.json"

for p in (HISTORY_PATH, INITIAL_PATH, SUMMARY_PATH):
    if not p.is_file():
        raise FileNotFoundError(p)

history = pd.read_csv(HISTORY_PATH)
initial = json.loads(INITIAL_PATH.read_text(encoding="utf-8"))
summary = json.loads(SUMMARY_PATH.read_text(encoding="utf-8"))

if len(history) < 2:
    raise RuntimeError(f"expected E1/E2 history, got {len(history)} rows")

diagnostic_keys = [
    "proxy/robust_mean_stage3_score",
    "proxy/robust_mean_accel_macro_f1",
    "proxy/robust_mean_steer_macro_f1",
    "proxy/sensitive/accel_macro_f1",
    "proxy/medium/accel_macro_f1",
    "proxy/conservative/accel_macro_f1",
    "proxy/medium/f1_accel_ACCELERATING",
    "proxy/medium/f1_accel_DECELERATING",
    "proxy/medium/f1_accel_CONSTANT",
    "proxy/medium/f1_accel_STOPPED",
    "proxy/medium/dynamic_to_constant_rate",
    "diag/accel/correlation",
    "diag/accel/pred_to_gt_std_ratio",
    "diag/accel/pred_vs_gt_slope",
    "diag/accel/pred_abs_p95_mps2",
    "reg/accel_from_speed_mps2/mae",
    "reg/accel_from_speed_mps2/rmse",
    "aux/ordinal/mean_accel_f1",
    "aux/ordinal/mean_decel_f1",
    "aux/ordinal/mean_three_state_macro_f1",
    "v5d/accel_state_ce",
    "v5d/accel_temporal_gradient",
    "v5d/integrated_kinematics",
    "total",
]

rows = []
for key in diagnostic_keys:
    col = f"val/{key}"
    if col not in history.columns:
        continue
    anchor = float(initial[key]) if key in initial else np.nan
    e1 = float(history.iloc[0][col])
    e2 = float(history.iloc[1][col])
    rows.append({
        "metric": key,
        "anchor": anchor,
        "E1": e1,
        "E2": e2,
        "E1-anchor": e1 - anchor if np.isfinite(anchor) else np.nan,
        "E2-anchor": e2 - anchor if np.isfinite(anchor) else np.nan,
        "E2-E1": e2 - e1,
    })

diagnostic_df = pd.DataFrame(rows)
display(diagnostic_df)

print("\nsummary.json")
print(json.dumps(summary, indent=2))


,metric,anchor,E1,E2,E1-anchor,E2-anchor,E2-E1
0,proxy/robust_mean_stage3_score,0.645956,0.649063,0.653489,0.003107,0.007533,0.004426
1,proxy/robust_mean_accel_macro_f1,0.646748,0.646775,0.651066,0.000026,0.004318,0.004291
2,proxy/robust_mean_steer_macro_f1,0.644106,0.654402,0.659142,0.010296,0.015036,0.004740
3,proxy/sensitive/accel_macro_f1,0.610033,0.614041,0.622171,0.004009,0.012138,0.008130
4,proxy/medium/accel_macro_f1,0.650767,0.652783,0.652523,0.002016,0.001756,-0.000260
5,proxy/conservative/accel_macro_f1,0.679446,0.673500,0.678504,-0.005946,-0.000942,0.005004
6,proxy/medium/f1_accel_ACCELERATING,0.456685,0.413162,0.436122,-0.043523,-0.020563,0.022959
7,proxy/medium/f1_accel_DECELERATING,0.454045,0.486500,0.466557,0.032455,0.012512,-0.019943
8,proxy/medium/f1_accel_CONSTANT,0.770998,0.771511,0.773721,0.000513,0.002723,0.002210
9,proxy/medium/f1_accel_STOPPED,0.921339,0.939959,0.933691,0.018620,0.012352,-0.006267



summary.json
{
  "run_variant": "vjepa21b_can_v5d_accel_decision",
  "git_commit": "afaf7e9",
  "vjepa_commit": "45d025f636dfc58fc2426905fc4a1ab755b1c3e5",
  "clip_len": 32,
  "warm_start_run": "vjepa21b_can_v5b_last2_ft",
  "warm_start_checkpoint": "best.pt",
  "finetune_block_indices": [
    10,
    11
  ],
  "diagnostic_best_epoch": 2,
  "diagnostic_best_proxy_stage3": 0.6534887754808312,
  "diagnostic_best_proxy_accel": 0.651065844927627,
  "diagnostic_best_proxy_steer": 0.6591422801049744,
  "initial_proxy_stage3": 0.6459556425824097,
  "initial_proxy_accel": 0.6467482889066564,
  "initial_proxy_steer": 0.6441061344925006,
  "delta_proxy_stage3_vs_warm_start": 0.007533132898421502,
  "delta_proxy_accel_vs_warm_start": 0.004317556020970614,
  "delta_proxy_steer_vs_warm_start": 0.015036145612473795,
  "peak_gpu_gib": 1.3543238639831543,
  "selection_checkpoint": "best_proxy.pt",
  "accel_selection_checkpoint": "best_accel_proxy.pt",
  "note": "proxy thresholds are diagnostic only; 

In [3]:
# Rule별로 acceleration 변화가 특정 threshold에서만 발생했는지 확인한다.
rule_rows = []
for rule_name in ("sensitive", "medium", "conservative"):
    metrics = [
        "accel_macro_f1",
        "dynamic_to_constant_rate",
        "f1_accel_ACCELERATING",
        "f1_accel_DECELERATING",
        "f1_accel_CONSTANT",
        "f1_accel_STOPPED",
    ]
    for metric in metrics:
        key = f"proxy/{rule_name}/{metric}"
        col = f"val/{key}"
        if key not in initial or col not in history.columns:
            continue
        anchor = float(initial[key])
        e1 = float(history.iloc[0][col])
        e2 = float(history.iloc[1][col])
        rule_rows.append({
            "rule": rule_name,
            "metric": metric,
            "anchor": anchor,
            "E1": e1,
            "E2": e2,
            "E1-anchor": e1 - anchor,
            "E2-anchor": e2 - anchor,
        })

rule_df = pd.DataFrame(rule_rows)
display(rule_df)

# checkpoint objective상 예상 epoch를 history만으로 먼저 계산한다.
objective_epoch = {
    "best.pt (min val total)": int(
        history.loc[pd.to_numeric(history["val/total"]).idxmin(), "epoch"]
    ),
    "best_proxy.pt (max robust stage3)": int(
        history.loc[
            pd.to_numeric(history["val/proxy/robust_mean_stage3_score"]).idxmax(),
            "epoch",
        ]
    ),
    "best_accel_proxy.pt (max robust accel)": int(
        history.loc[
            pd.to_numeric(history["val/proxy/robust_mean_accel_macro_f1"]).idxmax(),
            "epoch",
        ]
    ),
}
print(objective_epoch)


,rule,metric,anchor,E1,E2,E1-anchor,E2-anchor
0,sensitive,accel_macro_f1,0.610033,0.614041,0.622171,0.004009,0.012138
1,sensitive,dynamic_to_constant_rate,0.462602,0.441546,0.483162,-0.021056,0.020560
2,sensitive,f1_accel_ACCELERATING,0.473315,0.406130,0.459345,-0.067186,-0.013971
3,sensitive,f1_accel_DECELERATING,0.479249,0.511807,0.485306,0.032558,0.006057
4,sensitive,f1_accel_CONSTANT,0.612259,0.605036,0.624467,-0.007223,0.012208
5,sensitive,f1_accel_STOPPED,0.875307,0.933192,0.919565,0.057885,0.044258
6,medium,accel_macro_f1,0.650767,0.652783,0.652523,0.002016,0.001756
7,medium,dynamic_to_constant_rate,0.587728,0.590702,0.601110,0.002974,0.013382
8,medium,f1_accel_ACCELERATING,0.456685,0.413162,0.436122,-0.043523,-0.020563
9,medium,f1_accel_DECELERATING,0.454045,0.486500,0.466557,0.032455,0.012512


{'best.pt (min val total)': 2, 'best_proxy.pt (max robust stage3)': 2, 'best_accel_proxy.pt (max robust accel)': 2}


## 2. 실제 checkpoint epoch / SHA256 확인

세 checkpoint가 모두 E2를 가리키는 것으로 history상 예상되더라도,
실제 파일을 **byte fingerprint + checkpoint metadata**로 검증한다.

같은 SHA256이면 이후 overlap inference를 중복 실행하지 않는다.

In [4]:
def is_usable(path: Path, min_bytes: int = 1) -> bool:
    try:
        return path.is_file() and path.stat().st_size >= min_bytes
    except OSError:
        return False

def copy_to_local(source: Path, dest: Path, min_bytes: int = 1):
    dest.parent.mkdir(parents=True, exist_ok=True)
    tmp = dest.with_name(dest.name + ".tmp")
    tmp.unlink(missing_ok=True)
    with source.open("rb") as src, tmp.open("wb") as dst:
        shutil.copyfileobj(src, dst, length=16 * 1024 * 1024)
    if tmp.stat().st_size < min_bytes:
        raise OSError(f"staged file too small: {tmp.stat().st_size}")
    os.replace(tmp, dest)

def sha256_file(path: Path, chunk_size: int = 16 * 1024 * 1024) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        while True:
            chunk = f.read(chunk_size)
            if not chunk:
                break
            h.update(chunk)
    return h.hexdigest()

LOCAL_V5D_CKPT = Path("/content/v5d_checkpoints")
LOCAL_V5D_CKPT.mkdir(parents=True, exist_ok=True)

checkpoint_names = ["best.pt", "best_proxy.pt", "best_accel_proxy.pt"]
fingerprints = []

for name in checkpoint_names:
    src = V5D_RUN / name
    if not is_usable(src, 1_000_000):
        raise FileNotFoundError(src)

    dst = LOCAL_V5D_CKPT / name
    if (
        not is_usable(dst, src.stat().st_size)
        or dst.stat().st_size != src.stat().st_size
    ):
        copy_to_local(src, dst, min_bytes=1_000_000)

    fingerprints.append({
        "checkpoint": name,
        "bytes": dst.stat().st_size,
        "sha256": sha256_file(dst),
        "local_path": str(dst),
    })

fingerprint_df = pd.DataFrame(fingerprints)
display(fingerprint_df)

fingerprint_df.to_csv(
    REPORT_DIR / "checkpoint_fingerprints.csv",
    index=False,
)

print("unique checkpoint bytes:", fingerprint_df["sha256"].nunique())


,checkpoint,bytes,sha256,local_path
0,best.pt,519107465,7386360f3ee917e2d388024b0f8a482a62709e00699224...,/content/v5d_checkpoints/best.pt
1,best_proxy.pt,519107465,7386360f3ee917e2d388024b0f8a482a62709e00699224...,/content/v5d_checkpoints/best_proxy.pt
2,best_accel_proxy.pt,519107465,7386360f3ee917e2d388024b0f8a482a62709e00699224...,/content/v5d_checkpoints/best_accel_proxy.pt


unique checkpoint bytes: 1


## 3. v5-C protocol lock

새 validation split을 만들지 않는다.

가능하면 v5-C notebook이 저장한 다음 파일을 그대로 읽는다.

- `selected_segments.csv`
- `v5c_selection.json`
- 선택 overlap의 cached dense feature CSV

따라서 v5-D와 v5-C 비교에서 segment 구성, tune/holdout split,
decision proxy rule, overlap, v5-C fusion이 모두 동일하다.

In [5]:
V5C_SELECTION_PATH = V5C_RUN / "v5c_selection.json"
V5C_SEGMENTS_PATH = V5C_RUN / "selected_segments.csv"

if not V5C_SELECTION_PATH.is_file():
    raise FileNotFoundError(
        f"{V5C_SELECTION_PATH}\n"
        "Run notebook 15 through the 'Save v5-C selection' cell first."
    )
if not V5C_SEGMENTS_PATH.is_file():
    raise FileNotFoundError(
        f"{V5C_SEGMENTS_PATH}\n"
        "Run notebook 15 through the segment-selection cell first."
    )

v5c_selection = json.loads(
    V5C_SELECTION_PATH.read_text(encoding="utf-8")
)
selected = pd.read_csv(V5C_SEGMENTS_PATH)

OVERLAP_STRIDE = int(v5c_selection["overlap"]["stride"])
CENTER_FLOOR = float(v5c_selection["overlap"]["center_floor"])
CLIP_LEN = int(v5c_selection["clip_len"])
production_fusion = FusionConfig.from_mapping(
    v5c_selection["fusion"]
)

# Current locked v5-C result should be T32 / stride8 / center_floor .25.
assert CLIP_LEN == 32, CLIP_LEN
assert OVERLAP_STRIDE == 8, OVERLAP_STRIDE
assert abs(CENTER_FLOOR - 0.25) < 1e-12, CENTER_FLOOR
assert abs(float(production_fusion.accel_weight)) < 1e-12, (
    "v5-C lock changed: existing accel ordinal fusion is no longer zero"
)

proxy_rules = v5c_cfg["validation"]["proxy_rules"]

mc = v5d_cfg["model"]
STOP_THRESH = [float(x) for x in mc["stop_thresholds_mps"]]
ACCEL_THRESH = [float(x) for x in mc["accel_ordinal_thresholds_mps2"]]
STATE_THRESH = [float(x) for x in mc["accel_state"]["thresholds_mps2"]]
TURN_THRESH = [float(x) for x in mc["turn_yaw_thresholds_rps"]]

print("segments       :", len(selected))
print("split counts   :", selected["split"].value_counts().to_dict())
print("overlap        :", (CLIP_LEN, OVERLAP_STRIDE, CENTER_FLOOR))
print("v5-C fusion    :", production_fusion.as_dict())
print("state thresholds:", STATE_THRESH)


segments       : 25
split counts   : {'tune': 20, 'holdout': 5}
overlap        : (32, 8, 0.25)
v5-C fusion    : {'stop_weight': 0.75, 'accel_weight': 0.0, 'steer_weight': 0.3, 'turn_weight': 0.3, 'stop_temperature_mps': 0.25, 'accel_temperature_mps2': 0.1, 'steer_temperature_deg': 2.0}
state thresholds: [0.1, 0.2, 0.3, 0.5]


In [6]:
# v5-C의 이미 계산된 동일 overlap cache를 reference로 사용한다.
V5C_FEATURE_PATH = (
    V5C_RUN
    / "features"
    / f"features_s{OVERLAP_STRIDE}_c{CENTER_FLOOR:.2f}.csv"
)

if not V5C_FEATURE_PATH.is_file():
    raise FileNotFoundError(
        f"{V5C_FEATURE_PATH}\n"
        "Notebook 15의 overlap feature cell을 다시 실행하면 이 cache가 생성됩니다."
    )

v5c_features = pd.read_csv(V5C_FEATURE_PATH)

expected_keys = set(selected["segment_key"].astype(str))
found_keys = set(v5c_features["segment_key"].astype(str))
if expected_keys != found_keys:
    raise RuntimeError(
        f"v5-C cache segment mismatch: expected={len(expected_keys)}, "
        f"found={len(found_keys)}"
    )

def subset_for(table: pd.DataFrame, split: str) -> pd.DataFrame:
    keys = set(
        selected.loc[selected["split"] == split, "segment_key"].astype(str)
    )
    return table[
        table["segment_key"].astype(str).isin(keys)
    ].reset_index(drop=True)

def score_v5c_style(table: pd.DataFrame) -> dict[str, float]:
    return score_proxy_table(
        table,
        proxy_rules,
        fusion=production_fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
    )

v5c_scores = {
    "tune": score_v5c_style(subset_for(v5c_features, "tune")),
    "holdout": score_v5c_style(subset_for(v5c_features, "holdout")),
    "full": score_v5c_style(v5c_features),
}

print(json.dumps({
    split: {
        "stage3": d["proxy/robust_mean_stage3_score"],
        "accel": d["proxy/robust_mean_accel_macro_f1"],
        "steer": d["proxy/robust_mean_steer_macro_f1"],
        "accel_corr": d.get("diag/accel/correlation"),
        "accel_std_ratio": d.get("diag/accel/pred_to_gt_std_ratio"),
    }
    for split, d in v5c_scores.items()
}, indent=2))


{
  "tune": {
    "stage3": 0.7126309103781802,
    "accel": 0.7310800322768541,
    "steer": 0.6695829592812741,
    "accel_corr": 0.7181193979238295,
    "accel_std_ratio": 0.679080816605113
  },
  "holdout": {
    "stage3": 0.6495224068992792,
    "accel": 0.6891284467394773,
    "steer": 0.557108313938817,
    "accel_corr": 0.6655464997995674,
    "accel_std_ratio": 0.6866190259401919
  },
  "full": {
    "stage3": 0.7029045075876428,
    "accel": 0.7257382468068047,
    "steer": 0.6496257827429318,
    "accel_corr": 0.7127454254908605,
    "accel_std_ratio": 0.6802209899726198
  }
}


## 4. v5-D inference model 준비

v5-D는 v5-B checkpoint에서 warm-start한 뒤 마지막 2개 V-JEPA block을
fine-tune했으므로, v5-D checkpoint를 strict하게 load할 수 있는 동일 architecture를 만든다.

추론은 v5-C와 동일하게 FP32(`use_amp=False`)를 사용한다.

In [7]:
VJEPA_REPO = Path("/content/vjepa2")
VJEPA_COMMIT = "45d025f636dfc58fc2426905fc4a1ab755b1c3e5"

if not (VJEPA_REPO / ".git").is_dir():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/facebookresearch/vjepa2.git",
         str(VJEPA_REPO)],
        check=True,
    )

subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "fetch", "--all", "--tags"],
    check=True,
)
subprocess.run(
    ["git", "-C", str(VJEPA_REPO), "checkout", "-q", VJEPA_COMMIT],
    check=True,
)

VJEPA_NAME = "vjepa2_1_vitb_dist_vitG_384.pt"
VJEPA_DRIVE = PRETRAINED_ROOT / VJEPA_NAME
VJEPA_LOCAL = LOCAL_PRETRAINED_ROOT / VJEPA_NAME

if not is_usable(VJEPA_LOCAL, 1_000_000_000):
    if not is_usable(VJEPA_DRIVE, 1_000_000_000):
        raise FileNotFoundError(VJEPA_DRIVE)
    copy_to_local(VJEPA_DRIVE, VJEPA_LOCAL, min_bytes=1_000_000_000)

from blackbox_detection.stage3.vjepa21 import load_vjepa21_base_encoder
from blackbox_detection.stage3.v5d_accel import VJEPA21DenseCANV5D

dc = v5d_cfg["data"]
fusion_model_cfg = dict(mc["accel_fusion"])
spatial_cfg = dict(mc["spatial_pool"])
state_cfg = dict(mc["accel_state"])

backbone = load_vjepa21_base_encoder(
    VJEPA_REPO,
    VJEPA_LOCAL,
    num_frames=CLIP_LEN,
    out_layers=tuple(mc["out_layers"]),
    freeze=False,
)

model = VJEPA21DenseCANV5D(
    backbone,
    feature_dim=int(mc["feature_dim"]),
    temporal_hidden=int(mc["temporal_hidden"]),
    temporal_layers=int(mc["temporal_layers"]),
    spatial_grid=tuple(spatial_cfg["grid"]),
    spatial_gate_init=float(spatial_cfg["gate_init"]),
    accel_ordinal_thresholds_mps2=mc["accel_ordinal_thresholds_mps2"],
    accel_fusion_enabled=bool(fusion_model_cfg["enabled"]),
    accel_fusion_hidden=int(fusion_model_cfg["hidden"]),
    accel_fusion_gate_init=float(fusion_model_cfg["gate_init"]),
    accel_fusion_detach_ordinal_inputs=bool(
        fusion_model_cfg["detach_ordinal_inputs"]
    ),
    accel_state_thresholds_mps2=state_cfg["thresholds_mps2"],
    accel_state_hidden=int(state_cfg["hidden"]),
    accel_state_dropout=float(state_cfg["dropout"]),
    stop_thresholds_mps=mc["stop_thresholds_mps"],
    turn_yaw_thresholds_rps=mc["turn_yaw_thresholds_rps"],
    steer_activity_thresholds=mc["steer_activity_thresholds"],
    brake_thresholds_bar=mc["brake_thresholds_bar"],
    throttle_thresholds_pct=mc["throttle_thresholds_pct"],
)

ft_cfg = mc["backbone_finetune"]
ft_report = model.configure_partial_backbone(
    last_n_blocks=int(ft_cfg["last_n_blocks"]),
    train_final_tap_norm=bool(ft_cfg["train_final_tap_norm"]),
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device).eval()
model.requires_grad_(False)

print(ft_report)
print("device:", device)


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


BackboneFinetuneReport(depth=12, trainable_block_indices=(10, 11), trainable_norm_indices=(3,), trainable_backbone_params=14177280, frozen_backbone_params=72655872)
device: cuda


## 5. v5-D fixed overlap extractor

v5-C의 overlap-add 로직을 그대로 사용하면서 `accel_state_logits`의 softmax probability만
추가로 aggregate한다.

새 컬럼:
- `state_decel_p_*`
- `state_constant_p_*`
- `state_accel_p_*`

나머지 feature column 이름은 v5-C와 동일하다.

In [8]:
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)

def denorm(values: np.ndarray, name: str) -> np.ndarray:
    st = stats[name]
    return (
        values.astype(np.float32, copy=False) * float(st["std"])
        + float(st["mean"])
    )

@torch.inference_mode()
def infer_resized_frames_v5d_fixed(
    model: torch.nn.Module,
    frames_rgb_uint8: np.ndarray,
    *,
    batch_size: int = 2,
) -> pd.DataFrame:
    if frames_rgb_uint8.ndim != 4 or frames_rgb_uint8.shape[-1] != 3:
        raise ValueError(frames_rgb_uint8.shape)

    n = int(frames_rgb_uint8.shape[0])
    starts = window_starts(n, CLIP_LEN, OVERLAP_STRIDE)
    w_full = center_weights(CLIP_LEN, floor=CENTER_FLOOR).astype(np.float64)

    stop_k = len(STOP_THRESH)
    accel_k = len(ACCEL_THRESH)
    turn_k = len(TURN_THRESH)
    state_k = len(STATE_THRESH)

    acc = {
        "weight": np.zeros(n, dtype=np.float64),
        "speed_mps": np.zeros(n, dtype=np.float64),
        "accel_mps2": np.zeros(n, dtype=np.float64),
        "accel_raw_mps2": np.zeros(n, dtype=np.float64),
        "steering_deg": np.zeros(n, dtype=np.float64),
        "yaw_rate_rps": np.zeros(n, dtype=np.float64),
        "stop_prob": np.zeros((n, stop_k), dtype=np.float64),
        "accel_prob": np.zeros((n, accel_k, 2), dtype=np.float64),
        "steer_prob": np.zeros((n, 3), dtype=np.float64),
        "turn_prob": np.zeros((n, turn_k, 2), dtype=np.float64),
        "state_prob": np.zeros((n, state_k, 3), dtype=np.float64),
    }

    mean = torch.tensor(
        IMAGENET_MEAN, dtype=torch.float32
    )[None, :, None, None, None]
    std = torch.tensor(
        IMAGENET_STD, dtype=torch.float32
    )[None, :, None, None, None]

    for batch_start in range(0, len(starts), int(batch_size)):
        batch_starts = starts[batch_start: batch_start + int(batch_size)]
        clips = []
        valid_lengths = []

        for start in batch_starts:
            end = min(start + CLIP_LEN, n)
            clip = frames_rgb_uint8[start:end]
            valid_len = len(clip)
            if valid_len <= 0:
                raise RuntimeError("empty clip generated")
            if valid_len < CLIP_LEN:
                pad = np.repeat(clip[-1:], CLIP_LEN - valid_len, axis=0)
                clip = np.concatenate([clip, pad], axis=0)
            clips.append(clip)
            valid_lengths.append(valid_len)

        x = (
            torch.from_numpy(np.stack(clips, axis=0))
            .permute(0, 4, 1, 2, 3)
            .float()
            .div_(255.0)
        )
        x = ((x - mean) / std).to(device, non_blocking=True)

        # v5-C validation used use_amp=False.
        out = model(x)

        speed = denorm(
            out["speed_mps"].float().cpu().numpy(),
            "speed_mps",
        )
        accel = denorm(
            out["accel_from_speed_mps2"].float().cpu().numpy(),
            "accel_from_speed_mps2",
        )
        raw_norm = out.get(
            "accel_raw_from_speed_mps2",
            out["accel_from_speed_mps2"],
        )
        raw = denorm(
            raw_norm.float().cpu().numpy(),
            "accel_from_speed_mps2",
        )
        steering = denorm(
            out["steering_deg"].float().cpu().numpy(),
            "steering_deg",
        )
        yaw = denorm(
            out["yaw_rate_rps"].float().cpu().numpy(),
            "yaw_rate_rps",
        )

        stop_prob = torch.sigmoid(
            out["stop_ordinal_logits"]
        ).float().cpu().numpy()
        accel_prob = torch.sigmoid(
            out["accel_ordinal_logits"]
        ).float().cpu().numpy()
        steer_prob = torch.softmax(
            out["steer_direction_logits"].float(), dim=-1
        ).cpu().numpy()
        turn_prob = torch.sigmoid(
            out["turn_ordinal_logits"]
        ).float().cpu().numpy()
        state_prob = torch.softmax(
            out["accel_state_logits"].float(), dim=-1
        ).cpu().numpy()

        assert stop_prob.shape[-1] == stop_k
        assert accel_prob.shape[-2:] == (accel_k, 2)
        assert turn_prob.shape[-2:] == (turn_k, 2)
        assert state_prob.shape[-2:] == (state_k, 3)

        for j, (start, valid_len) in enumerate(
            zip(batch_starts, valid_lengths, strict=True)
        ):
            sl = slice(start, start + valid_len)
            w = w_full[:valid_len]
            acc["weight"][sl] += w
            acc["speed_mps"][sl] += speed[j, :valid_len] * w
            acc["accel_mps2"][sl] += accel[j, :valid_len] * w
            acc["accel_raw_mps2"][sl] += raw[j, :valid_len] * w
            acc["steering_deg"][sl] += steering[j, :valid_len] * w
            acc["yaw_rate_rps"][sl] += yaw[j, :valid_len] * w
            acc["stop_prob"][sl] += stop_prob[j, :valid_len] * w[:, None]
            acc["accel_prob"][sl] += (
                accel_prob[j, :valid_len] * w[:, None, None]
            )
            acc["steer_prob"][sl] += (
                steer_prob[j, :valid_len] * w[:, None]
            )
            acc["turn_prob"][sl] += (
                turn_prob[j, :valid_len] * w[:, None, None]
            )
            acc["state_prob"][sl] += (
                state_prob[j, :valid_len] * w[:, None, None]
            )

    weight = acc.pop("weight")
    if np.any(weight <= 0):
        raise RuntimeError("some frames received zero overlap weight")

    d1 = weight
    d2 = weight[:, None]
    d3 = weight[:, None, None]

    frame = pd.DataFrame({
        "sample_index": np.arange(n, dtype=np.int64),
        "speed_mps": acc["speed_mps"] / d1,
        "accel_mps2": acc["accel_mps2"] / d1,
        "accel_raw_mps2": acc["accel_raw_mps2"] / d1,
        "steering_deg": acc["steering_deg"] / d1,
        "yaw_rate_rps": acc["yaw_rate_rps"] / d1,
    })

    stop_avg = acc["stop_prob"] / d2
    accel_avg = acc["accel_prob"] / d3
    steer_avg = acc["steer_prob"] / d2
    turn_avg = acc["turn_prob"] / d3
    state_avg = acc["state_prob"] / d3

    for k, thr in enumerate(STOP_THRESH):
        frame[f"stop_p_{threshold_tag(thr)}"] = stop_avg[:, k]

    for k, thr in enumerate(ACCEL_THRESH):
        tag = threshold_tag(thr)
        frame[f"accel_decel_p_{tag}"] = accel_avg[:, k, 0]
        frame[f"accel_accel_p_{tag}"] = accel_avg[:, k, 1]

    frame["steer_p_left"] = steer_avg[:, 0]
    frame["steer_p_straight"] = steer_avg[:, 1]
    frame["steer_p_right"] = steer_avg[:, 2]

    for k, thr in enumerate(TURN_THRESH):
        tag = threshold_tag(thr)
        frame[f"turn_neg_p_{tag}"] = turn_avg[:, k, 0]
        frame[f"turn_pos_p_{tag}"] = turn_avg[:, k, 1]

    for k, thr in enumerate(STATE_THRESH):
        tag = threshold_tag(thr)
        frame[f"state_decel_p_{tag}"] = state_avg[:, k, 0]
        frame[f"state_constant_p_{tag}"] = state_avg[:, k, 1]
        frame[f"state_accel_p_{tag}"] = state_avg[:, k, 2]

    return frame

def extract_video_features_v5d_fixed(
    model: torch.nn.Module,
    video_path: str | Path,
    *,
    raw_stride: int = 1,
    raw_offset: int = 0,
    batch_size: int = 2,
) -> pd.DataFrame:
    frames = decode_sampled_video(
        video_path,
        input_height=int(dc["input_height"]),
        input_width=int(dc["input_width"]),
        raw_stride=int(raw_stride),
        raw_offset=int(raw_offset),
    )
    return infer_resized_frames_v5d_fixed(
        model,
        frames,
        batch_size=batch_size,
    )

def attach_truth(pred: pd.DataFrame, row) -> pd.DataFrame:
    meta = read_frame_table(
        COMMA_ROOT / row.metadata_relpath
    ).reset_index(drop=True)

    if len(meta) != len(pred):
        raise RuntimeError(
            f"frame count mismatch {row.segment_key}: "
            f"video={len(pred)}, metadata={len(meta)}"
        )

    out = pred.copy()
    out.insert(0, "segment_key", str(row.segment_key))
    out.insert(1, "route_id", str(row.route_id))
    out.insert(2, "segment_id", str(row.segment_id))
    out["gt_speed_mps"] = meta["speed_mps"].to_numpy(dtype=np.float32)
    out["gt_accel_mps2"] = meta[
        "accel_from_speed_mps2"
    ].to_numpy(dtype=np.float32)
    out["gt_steering_deg"] = meta[
        "steering_deg"
    ].to_numpy(dtype=np.float32)
    out["valid_speed"] = meta["valid_speed"].to_numpy(dtype=bool)
    out["valid_accel"] = meta[
        "valid_accel_from_speed"
    ].to_numpy(dtype=bool)
    out["valid_steer"] = meta["valid_steer"].to_numpy(dtype=bool)
    return out


## 6. checkpoint별 동일 overlap inference

SHA256이 같은 checkpoint는 한 번만 추론한다.
각 unique checkpoint의 metadata epoch도 여기서 strict load로 확인한다.

In [9]:
V5D_FEATURE_DIR = REPORT_DIR / "features"
V5D_FEATURE_DIR.mkdir(parents=True, exist_ok=True)

checkpoint_tables = {}
checkpoint_meta = {}
sha_to_table = {}

for row in fingerprint_df.itertuples(index=False):
    name = str(row.checkpoint)
    sha = str(row.sha256)
    local_path = Path(row.local_path)

    if sha in sha_to_table:
        representative = sha_to_table[sha]["representative"]
        checkpoint_tables[name] = sha_to_table[sha]["table"]
        checkpoint_meta[name] = {
            **checkpoint_meta[representative],
            "deduplicated_from": representative,
        }
        print(name, "-> same bytes as", representative)
        continue

    meta = load_checkpoint(
        local_path,
        model=model,
        optimizer=None,
        scheduler=None,
        map_location="cpu",
        strict=True,
        restore_rng_state=False,
    )
    model.to(device).eval()
    model.requires_grad_(False)

    checkpoint_meta[name] = {
        "epoch": meta.get("epoch"),
        "sha256": sha,
        "deduplicated_from": None,
    }

    cache = (
        V5D_FEATURE_DIR
        / f"{name.replace('.pt','')}__{sha[:12]}"
        f"__s{OVERLAP_STRIDE}_c{CENTER_FLOOR:.2f}.csv"
    )

    if cache.is_file():
        print(name, "CACHE HIT:", cache)
        table = pd.read_csv(cache)
    else:
        parts = []
        t0 = time.perf_counter()

        for i, seg in enumerate(selected.itertuples(index=False), start=1):
            pred = extract_video_features_v5d_fixed(
                model,
                COMMA_ROOT / seg.video_relpath,
                raw_stride=1,
                raw_offset=0,
                batch_size=int(v5c_cfg["data"]["batch_size"]),
            )
            parts.append(attach_truth(pred, seg))

            if i == 1 or i % 5 == 0 or i == len(selected):
                print(f"{name}: {i}/{len(selected)} segments")

        table = pd.concat(parts, ignore_index=True)
        table.to_csv(cache, index=False)
        print(
            name,
            "saved",
            cache,
            "minutes=",
            (time.perf_counter() - t0) / 60.0,
        )

    found = set(table["segment_key"].astype(str))
    if found != expected_keys:
        raise RuntimeError(f"{name}: segment set mismatch")

    checkpoint_tables[name] = table
    sha_to_table[sha] = {
        "representative": name,
        "table": table,
    }

# Resolve deduplicated entries to the representative table.
for name in checkpoint_names:
    sha = str(
        fingerprint_df.loc[
            fingerprint_df["checkpoint"] == name, "sha256"
        ].iloc[0]
    )
    checkpoint_tables[name] = sha_to_table[sha]["table"]

print(json.dumps(checkpoint_meta, indent=2, default=str))


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)


best.pt: 1/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

best.pt: 5/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

...

best.pt: 15/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

best.pt: 20/25 segments


/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context manager will be removed. Please see `torch.nn.attention.sdpa_kernel()` for the new context manager, with updated signature.
  self.gen = func(*args, **kwds)
/usr/lib/python3.12/contextlib.py:105: FutureWarning: `torch.backends.cuda.sdp_kernel()` is deprecated. In the future, this context m

best.pt: 25/25 segments
best.pt saved /content/drive/MyDrive/Blackbox-Detection/outputs/stage3/vjepa21b_can_v5d_accel_decision/overlap_validation/features/best__7386360f3ee9__s8_c0.25.csv minutes= 14.384295990816666
best_proxy.pt -> same bytes as best.pt
best_accel_proxy.pt -> same bytes as best.pt
{
  "best.pt": {
    "epoch": 2,
    "sha256": "7386360f3ee917e2d388024b0f8a482a62709e006992244ad267d937528f5615",
    "deduplicated_from": null
  },
  "best_proxy.pt": {
    "epoch": 2,
    "sha256": "7386360f3ee917e2d388024b0f8a482a62709e006992244ad267d937528f5615",
    "deduplicated_from": "best.pt"
  },
  "best_accel_proxy.pt": {
    "epoch": 2,
    "sha256": "7386360f3ee917e2d388024b0f8a482a62709e006992244ad267d937528f5615",
    "deduplicated_from": "best.pt"
  }
}


## 7. v5-C production inference와 checkpoint별 v5-D 비교

아직 새 3-state head를 쓰지 않는다.

즉 **v5-D continuous/기존 aux output + v5-C production fusion**만 적용한다.
이 비교가 먼저 좋아져야 학습 자체의 acceleration 표현 개선이라고 볼 수 있다.

In [10]:
comparison_rows = []

# v5-C reference
for split_name in ("tune", "holdout", "full"):
    score = v5c_scores[split_name]
    comparison_rows.append({
        "model": "v5-C / v5-B checkpoint",
        "checkpoint": v5c_selection["source_checkpoint"],
        "split": split_name,
        "stage3": score["proxy/robust_mean_stage3_score"],
        "accel": score["proxy/robust_mean_accel_macro_f1"],
        "steer": score["proxy/robust_mean_steer_macro_f1"],
        "accel_corr": score.get("diag/accel/correlation"),
        "accel_std_ratio": score.get("diag/accel/pred_to_gt_std_ratio"),
    })

for name, table in checkpoint_tables.items():
    parts = {
        "tune": subset_for(table, "tune"),
        "holdout": subset_for(table, "holdout"),
        "full": table,
    }
    for split_name, part in parts.items():
        score = score_v5c_style(part)
        comparison_rows.append({
            "model": "v5-D",
            "checkpoint": name,
            "split": split_name,
            "stage3": score["proxy/robust_mean_stage3_score"],
            "accel": score["proxy/robust_mean_accel_macro_f1"],
            "steer": score["proxy/robust_mean_steer_macro_f1"],
            "accel_corr": score.get("diag/accel/correlation"),
            "accel_std_ratio": score.get("diag/accel/pred_to_gt_std_ratio"),
        })

comparison_df = pd.DataFrame(comparison_rows)

ref_full = comparison_df[
    (comparison_df["model"] == "v5-C / v5-B checkpoint")
    & (comparison_df["split"] == "full")
].iloc[0]

comparison_df["delta_accel_vs_v5c_full"] = np.where(
    comparison_df["split"].eq("full"),
    comparison_df["accel"] - float(ref_full["accel"]),
    np.nan,
)
comparison_df["delta_stage3_vs_v5c_full"] = np.where(
    comparison_df["split"].eq("full"),
    comparison_df["stage3"] - float(ref_full["stage3"]),
    np.nan,
)

display(comparison_df)
comparison_df.to_csv(
    REPORT_DIR / "overlap_checkpoint_comparison.csv",
    index=False,
)

# State fusion 실험에 사용할 checkpoint:
# tune acceleration이 가장 높은 것을 고르되 checkpoint 이름으로 tie-break한다.
candidate_rows = comparison_df[
    (comparison_df["model"] == "v5-D")
    & (comparison_df["split"] == "tune")
].copy()

priority = {
    "best_accel_proxy.pt": 0,
    "best_proxy.pt": 1,
    "best.pt": 2,
}
candidate_rows["priority"] = candidate_rows["checkpoint"].map(priority).fillna(99)
candidate_rows = candidate_rows.sort_values(
    ["accel", "priority"],
    ascending=[False, True],
)
STATE_CKPT = str(candidate_rows.iloc[0]["checkpoint"])
state_features = checkpoint_tables[STATE_CKPT]

print("state-fusion checkpoint:", STATE_CKPT)


,model,checkpoint,split,stage3,accel,steer,accel_corr,accel_std_ratio,delta_accel_vs_v5c_full,delta_stage3_vs_v5c_full
0,v5-C / v5-B checkpoint,best.pt,tune,0.712631,0.731080,0.669583,0.718119,0.679081,NaN,NaN
1,v5-C / v5-B checkpoint,best.pt,holdout,0.649522,0.689128,0.557108,0.665546,0.686619,NaN,NaN
2,v5-C / v5-B checkpoint,best.pt,full,0.702905,0.725738,0.649626,0.712745,0.680221,0.000000,0.000000
3,v5-D,best.pt,tune,0.717680,0.728775,0.691792,0.720738,0.682940,NaN,NaN
4,v5-D,best.pt,holdout,0.653020,0.695582,0.553706,0.685076,0.697551,NaN,NaN
5,v5-D,best.pt,full,0.707421,0.725305,0.665692,0.717253,0.684383,-0.000433,0.004517
6,v5-D,best_proxy.pt,tune,0.717680,0.728775,0.691792,0.720738,0.682940,NaN,NaN
7,v5-D,best_proxy.pt,holdout,0.653020,0.695582,0.553706,0.685076,0.697551,NaN,NaN
8,v5-D,best_proxy.pt,full,0.707421,0.725305,0.665692,0.717253,0.684383,-0.000433,0.004517
9,v5-D,best_accel_proxy.pt,tune,0.717680,0.728775,0.691792,0.720738,0.682940,NaN,NaN


state-fusion checkpoint: best_accel_proxy.pt


## 8. 3-state head fusion 정의

3-state head는 각 threshold에서 `DECEL / CONSTANT / ACCEL`을 직접 경쟁시킨다.

기존 v5-C continuous decision score에 다음 evidence만 더한다.

- ACCEL 쪽: `log P(ACCEL) - log P(CONSTANT)`
- DECEL 쪽: `log P(DECEL) - log P(CONSTANT)`

threshold는 현재 proxy rule의 deadzone에 맞춰
학습된 `[0.10, 0.20, 0.30, 0.50]` probability를 선형 interpolation한다.

**중요:** threshold, temperature, class bias는 탐색하지 않는다.
탐색 변수는 `state_weight` 하나뿐이다.

In [11]:
def _clip_prob(x):
    return np.clip(np.asarray(x, dtype=np.float64), 1e-5, 1.0 - 1e-5)

def _logit(x):
    p = _clip_prob(x)
    return np.log(p) - np.log1p(-p)

def _interp_matrix(
    thresholds,
    matrix: np.ndarray,
    target: float,
) -> np.ndarray:
    thresholds = np.asarray(thresholds, dtype=np.float64)
    matrix = np.asarray(matrix, dtype=np.float64)
    if matrix.ndim != 2 or matrix.shape[1] != len(thresholds):
        raise ValueError((matrix.shape, len(thresholds)))

    t = float(target)
    if t <= thresholds[0]:
        return matrix[:, 0]
    if t >= thresholds[-1]:
        return matrix[:, -1]

    hi = int(np.searchsorted(thresholds, t, side="right"))
    lo = hi - 1
    alpha = (t - thresholds[lo]) / (thresholds[hi] - thresholds[lo])
    return (1.0 - alpha) * matrix[:, lo] + alpha * matrix[:, hi]

def _matrix_cols(frame, prefix, thresholds):
    cols = [
        f"{prefix}{threshold_tag(t)}"
        for t in thresholds
    ]
    missing = [c for c in cols if c not in frame.columns]
    if missing:
        raise KeyError(missing)
    return frame[cols].to_numpy(dtype=np.float64)

def labels_with_state_fusion(
    frame: pd.DataFrame,
    *,
    rule: DecisionRule,
    base_fusion: FusionConfig,
    state_weight: float,
) -> tuple[np.ndarray, np.ndarray]:
    # Existing v5-C logic provides exact STOP and steering behavior.
    base_accel_labels, steer_labels = labels_with_aux_fusion(
        frame,
        rule=rule,
        fusion=base_fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
        accel_source="fused",
    )
    stopped = base_accel_labels == "STOPPED"

    accel = (
        frame["accel_mps2"].to_numpy(dtype=np.float64)
        + float(rule.accel_bias_mps2)
    )
    accel_temp = max(
        float(base_fusion.accel_temperature_mps2),
        1e-4,
    )

    z_acc = (
        accel - float(rule.accel_pos_mps2)
    ) / accel_temp
    z_dec = (
        -accel - float(rule.accel_neg_mps2)
    ) / accel_temp

    # Preserve existing ordinal acceleration fusion if a future v5-C lock
    # changes it. Current production value is 0.0.
    if abs(float(base_fusion.accel_weight)) > 1e-12:
        p_acc = _interp_matrix(
            ACCEL_THRESH,
            _matrix_cols(frame, "accel_accel_p_", ACCEL_THRESH),
            float(rule.accel_pos_mps2),
        )
        p_dec = _interp_matrix(
            ACCEL_THRESH,
            _matrix_cols(frame, "accel_decel_p_", ACCEL_THRESH),
            float(rule.accel_neg_mps2),
        )
        z_acc += float(base_fusion.accel_weight) * _logit(p_acc)
        z_dec += float(base_fusion.accel_weight) * _logit(p_dec)

    w = float(state_weight)
    if abs(w) > 1e-12:
        state_decel = _matrix_cols(
            frame, "state_decel_p_", STATE_THRESH
        )
        state_const = _matrix_cols(
            frame, "state_constant_p_", STATE_THRESH
        )
        state_accel = _matrix_cols(
            frame, "state_accel_p_", STATE_THRESH
        )

        p_acc = _interp_matrix(
            STATE_THRESH, state_accel, float(rule.accel_pos_mps2)
        )
        p_const_acc = _interp_matrix(
            STATE_THRESH, state_const, float(rule.accel_pos_mps2)
        )
        p_dec = _interp_matrix(
            STATE_THRESH, state_decel, float(rule.accel_neg_mps2)
        )
        p_const_dec = _interp_matrix(
            STATE_THRESH, state_const, float(rule.accel_neg_mps2)
        )

        z_acc += w * (
            np.log(_clip_prob(p_acc))
            - np.log(_clip_prob(p_const_acc))
        )
        z_dec += w * (
            np.log(_clip_prob(p_dec))
            - np.log(_clip_prob(p_const_dec))
        )

    accel_labels = np.full(len(frame), "CONSTANT", dtype=object)
    accel_labels[stopped] = "STOPPED"

    moving = ~stopped
    choose_acc = moving & (z_acc > 0.0) & (z_acc >= z_dec)
    choose_dec = moving & (z_dec > 0.0) & (z_dec > z_acc)
    accel_labels[choose_acc] = "ACCELERATING"
    accel_labels[choose_dec] = "DECELERATING"

    return accel_labels.astype(str), steer_labels.astype(str)

def truth_proxy_labels(
    frame: pd.DataFrame,
    rule: ProxyRule,
) -> tuple[np.ndarray, np.ndarray]:
    speed = frame["gt_speed_mps"].to_numpy(dtype=np.float64)
    accel = frame["gt_accel_mps2"].to_numpy(dtype=np.float64)
    steer = frame["gt_steering_deg"].to_numpy(dtype=np.float64)

    accel_labels = np.full(len(frame), "CONSTANT", dtype=object)
    stopped = speed <= float(rule.stop_speed_mps)
    moving = ~stopped
    accel_labels[stopped] = "STOPPED"
    accel_labels[
        moving & (accel > float(rule.accel_deadzone_mps2))
    ] = "ACCELERATING"
    accel_labels[
        moving & (accel < -float(rule.accel_deadzone_mps2))
    ] = "DECELERATING"

    steer_labels = np.full(len(frame), "STRAIGHT", dtype=object)
    steer_labels[
        steer < -float(rule.steer_deadzone_deg)
    ] = "LEFT"
    steer_labels[
        steer > float(rule.steer_deadzone_deg)
    ] = "RIGHT"

    return accel_labels.astype(str), steer_labels.astype(str)

def score_state_fusion(
    frame: pd.DataFrame,
    state_weight: float,
) -> dict[str, float]:
    required = {
        "gt_speed_mps",
        "gt_accel_mps2",
        "gt_steering_deg",
        "speed_mps",
        "accel_mps2",
        "accel_raw_mps2",
        "steering_deg",
    }
    valid = np.ones(len(frame), dtype=bool)
    for col in ("valid_speed", "valid_accel", "valid_steer"):
        if col in frame.columns:
            valid &= frame[col].to_numpy(dtype=bool)
    for col in required:
        valid &= np.isfinite(
            frame[col].to_numpy(dtype=np.float64)
        )

    work = frame.loc[valid].reset_index(drop=True)
    if work.empty:
        raise ValueError("no valid rows")

    stage3_scores = []
    accel_scores = []
    steer_scores = []
    result = {}

    for name, raw_rule in proxy_rules.items():
        prule = (
            raw_rule
            if isinstance(raw_rule, ProxyRule)
            else ProxyRule.from_mapping(raw_rule)
        )
        drule = DecisionRule.from_proxy(prule)

        truth_a, truth_s = truth_proxy_labels(work, prule)
        pred_a, pred_s = labels_with_state_fusion(
            work,
            rule=drule,
            base_fusion=production_fusion,
            state_weight=float(state_weight),
        )

        metrics = dacon_stage3_metrics(
            truth_a, pred_a, truth_s, pred_s
        )

        prefix = f"proxy/{name}"
        result[f"{prefix}/stage3_score"] = float(
            metrics["stage3_score"]
        )
        result[f"{prefix}/accel_macro_f1"] = float(
            metrics["accel_macro_f1"]
        )
        result[f"{prefix}/steer_macro_f1"] = float(
            metrics["steer_macro_f1"]
        )

        per_class = f1_score(
            truth_a,
            pred_a,
            labels=list(ACCEL_CLASSES),
            average=None,
            zero_division=0,
        )
        for cls, value in zip(
            ACCEL_CLASSES, per_class, strict=True
        ):
            result[f"{prefix}/f1_accel_{cls}"] = float(value)

        dynamic = np.isin(
            truth_a, ["ACCELERATING", "DECELERATING"]
        )
        if dynamic.any():
            result[
                f"{prefix}/dynamic_to_constant_rate"
            ] = float(np.mean(pred_a[dynamic] == "CONSTANT"))
        else:
            result[
                f"{prefix}/dynamic_to_constant_rate"
            ] = 0.0

        stage3_scores.append(float(metrics["stage3_score"]))
        accel_scores.append(float(metrics["accel_macro_f1"]))
        steer_scores.append(float(metrics["steer_macro_f1"]))

    result["proxy/robust_mean_stage3_score"] = float(
        np.mean(stage3_scores)
    )
    result["proxy/robust_min_stage3_score"] = float(
        np.min(stage3_scores)
    )
    result["proxy/robust_mean_accel_macro_f1"] = float(
        np.mean(accel_scores)
    )
    result["proxy/robust_min_accel_macro_f1"] = float(
        np.min(accel_scores)
    )
    result["proxy/robust_mean_steer_macro_f1"] = float(
        np.mean(steer_scores)
    )
    return result

# weight=0 must reproduce the existing v5-C scoring path exactly.
parity_old = score_v5c_style(state_features)
parity_new = score_state_fusion(state_features, state_weight=0.0)

for key in (
    "proxy/robust_mean_stage3_score",
    "proxy/robust_mean_accel_macro_f1",
    "proxy/robust_mean_steer_macro_f1",
):
    if abs(parity_old[key] - parity_new[key]) > 1e-10:
        raise AssertionError(
            f"state fusion parity failed for {key}: "
            f"{parity_old[key]} vs {parity_new[key]}"
        )

print("weight=0 parity: PASS")


weight=0 parity: PASS


## 9. state weight 1-D search — tune only

학습된 multi-threshold state head는 고정하고 `state_weight`만 탐색한다.

공개 50 labels는 절대 이 선택에 들어가지 않는다.

In [12]:
STATE_WEIGHTS = [0.0, 0.15, 0.30, 0.50, 0.75, 1.00]

tune_state = subset_for(state_features, "tune")
hold_state = subset_for(state_features, "holdout")

grid_rows = []
for weight in STATE_WEIGHTS:
    score = score_state_fusion(
        tune_state,
        state_weight=weight,
    )
    grid_rows.append({
        "state_weight": weight,
        "tune_stage3": score[
            "proxy/robust_mean_stage3_score"
        ],
        "tune_accel": score[
            "proxy/robust_mean_accel_macro_f1"
        ],
        "tune_min_accel": score[
            "proxy/robust_min_accel_macro_f1"
        ],
        "tune_medium_accelerating": score[
            "proxy/medium/f1_accel_ACCELERATING"
        ],
        "tune_medium_dynamic_to_constant": score[
            "proxy/medium/dynamic_to_constant_rate"
        ],
    })

state_grid = pd.DataFrame(grid_rows).sort_values(
    ["tune_accel", "tune_min_accel", "state_weight"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(state_grid)
state_grid.to_csv(
    REPORT_DIR / "state_fusion_grid.csv",
    index=False,
)

BEST_STATE_WEIGHT = float(
    state_grid.iloc[0]["state_weight"]
)
print("selected on tune only:", BEST_STATE_WEIGHT)


,state_weight,tune_stage3,tune_accel,tune_min_accel,tune_medium_accelerating,tune_medium_dynamic_to_constant
0,1.00,0.718614,0.730109,0.725653,0.523213,0.483598
1,0.75,0.718457,0.729885,0.724955,0.523213,0.483386
2,0.50,0.718103,0.729379,0.723724,0.522965,0.483175
3,0.00,0.717680,0.728775,0.720544,0.524095,0.480847
4,0.15,0.717664,0.728752,0.721078,0.523958,0.481270
5,0.30,0.717630,0.728704,0.721675,0.523710,0.481693


selected on tune only: 1.0


## 10. holdout / full gate

최종 local 판단은 v5-C reference와 직접 비교한다.

엄격한 acceleration gate:

- holdout robust accel: v5-C 이상
- full robust accel: v5-C 대비 최소 `+0.002`
- medium `ACCELERATING` F1: v5-C 이상
- medium `dynamic_to_constant_rate`: v5-C 이하
- 세 proxy rule 중 최악의 accel delta가 `-0.005`보다 나쁘지 않음

이 gate는 DACON hidden 성능을 보장하는 규칙이 아니라,
**현재 v5-D를 제출 후보로 올릴 최소 local evidence**다.

In [13]:
def eval_splits(table, state_weight):
    return {
        "tune": score_state_fusion(
            subset_for(table, "tune"),
            state_weight,
        ),
        "holdout": score_state_fusion(
            subset_for(table, "holdout"),
            state_weight,
        ),
        "full": score_state_fusion(
            table,
            state_weight,
        ),
    }

v5d_state_scores = eval_splits(
    state_features,
    BEST_STATE_WEIGHT,
)
v5d_no_state_scores = eval_splits(
    state_features,
    0.0,
)

# v5-C에는 state columns가 없으므로 기존 scorer의 결과를 사용한다.
def metric_row(label, split, score):
    return {
        "variant": label,
        "split": split,
        "stage3": score["proxy/robust_mean_stage3_score"],
        "accel": score["proxy/robust_mean_accel_macro_f1"],
        "steer": score["proxy/robust_mean_steer_macro_f1"],
        "medium_accelerating": score.get(
            "proxy/medium/f1_accel_ACCELERATING"
        ),
        "medium_decelerating": score.get(
            "proxy/medium/f1_accel_DECELERATING"
        ),
        "medium_constant": score.get(
            "proxy/medium/f1_accel_CONSTANT"
        ),
        "medium_stopped": score.get(
            "proxy/medium/f1_accel_STOPPED"
        ),
        "medium_dynamic_to_constant": score.get(
            "proxy/medium/dynamic_to_constant_rate"
        ),
    }

# v5-C score_proxy_table에는 dynamic_to_constant가 없으므로
# state scorer와 동일 truth/label 정의로 별도 계산하기 위해,
# v5-C reference에는 weight=0 state scorer를 쓸 수 없다.
# 대신 dynamic metric만 기존 production label에서 직접 계산한다.
def valid_proxy_rows(frame: pd.DataFrame) -> pd.DataFrame:
    required = {
        "gt_speed_mps",
        "gt_accel_mps2",
        "gt_steering_deg",
        "speed_mps",
        "accel_mps2",
        "accel_raw_mps2",
        "steering_deg",
    }
    valid = np.ones(len(frame), dtype=bool)
    for col in ("valid_speed", "valid_accel", "valid_steer"):
        if col in frame.columns:
            valid &= frame[col].to_numpy(dtype=bool)
    for col in required:
        valid &= np.isfinite(
            frame[col].to_numpy(dtype=np.float64)
        )
    return frame.loc[valid].reset_index(drop=True)

def v5c_dynamic_metric(frame, raw_rule):
    work = valid_proxy_rows(frame)
    prule = ProxyRule.from_mapping(raw_rule)
    drule = DecisionRule.from_proxy(prule)
    truth_a, truth_s = truth_proxy_labels(work, prule)
    pred_a, _ = labels_with_aux_fusion(
        work,
        rule=drule,
        fusion=production_fusion,
        stop_thresholds_mps=STOP_THRESH,
        accel_thresholds_mps2=ACCEL_THRESH,
        turn_thresholds_rps=TURN_THRESH,
    )
    dynamic = np.isin(
        truth_a, ["ACCELERATING", "DECELERATING"]
    )
    return (
        float(np.mean(pred_a[dynamic] == "CONSTANT"))
        if dynamic.any() else 0.0
    )

# v5-C full per-class metrics are computed explicitly for a fair gate.
def v5c_extra_metrics(frame):
    work = valid_proxy_rows(frame)
    out = {}
    for name, raw_rule in proxy_rules.items():
        prule = ProxyRule.from_mapping(raw_rule)
        drule = DecisionRule.from_proxy(prule)
        truth_a, truth_s = truth_proxy_labels(work, prule)
        pred_a, pred_s = labels_with_aux_fusion(
            work,
            rule=drule,
            fusion=production_fusion,
            stop_thresholds_mps=STOP_THRESH,
            accel_thresholds_mps2=ACCEL_THRESH,
            turn_thresholds_rps=TURN_THRESH,
        )
        vals = f1_score(
            truth_a,
            pred_a,
            labels=list(ACCEL_CLASSES),
            average=None,
            zero_division=0,
        )
        for cls, val in zip(ACCEL_CLASSES, vals, strict=True):
            out[f"proxy/{name}/f1_accel_{cls}"] = float(val)
        dynamic = np.isin(
            truth_a, ["ACCELERATING", "DECELERATING"]
        )
        out[f"proxy/{name}/dynamic_to_constant_rate"] = (
            float(np.mean(pred_a[dynamic] == "CONSTANT"))
            if dynamic.any() else 0.0
        )
    return out

v5c_extra = {
    "tune": v5c_extra_metrics(subset_for(v5c_features, "tune")),
    "holdout": v5c_extra_metrics(subset_for(v5c_features, "holdout")),
    "full": v5c_extra_metrics(v5c_features),
}

for split in ("tune", "holdout", "full"):
    v5c_scores[split].update(v5c_extra[split])

gate_table = pd.DataFrame(
    [
        metric_row("v5-C", split, v5c_scores[split])
        for split in ("tune", "holdout", "full")
    ]
    + [
        metric_row(
            f"v5-D {STATE_CKPT} state=0",
            split,
            v5d_no_state_scores[split],
        )
        for split in ("tune", "holdout", "full")
    ]
    + [
        metric_row(
            f"v5-D {STATE_CKPT} state={BEST_STATE_WEIGHT:g}",
            split,
            v5d_state_scores[split],
        )
        for split in ("tune", "holdout", "full")
    ]
)

display(gate_table)

ref_hold = v5c_scores["holdout"]
ref_full = v5c_scores["full"]
cand_hold = v5d_state_scores["holdout"]
cand_full = v5d_state_scores["full"]

rule_accel_deltas = {
    name: (
        cand_full[f"proxy/{name}/accel_macro_f1"]
        - ref_full[f"proxy/{name}/accel_macro_f1"]
    )
    for name in proxy_rules
}
worst_rule_accel_delta = min(rule_accel_deltas.values())

delta_hold_accel = (
    cand_hold["proxy/robust_mean_accel_macro_f1"]
    - ref_hold["proxy/robust_mean_accel_macro_f1"]
)
delta_full_accel = (
    cand_full["proxy/robust_mean_accel_macro_f1"]
    - ref_full["proxy/robust_mean_accel_macro_f1"]
)
delta_full_stage3 = (
    cand_full["proxy/robust_mean_stage3_score"]
    - ref_full["proxy/robust_mean_stage3_score"]
)
delta_medium_acc = (
    cand_full["proxy/medium/f1_accel_ACCELERATING"]
    - ref_full["proxy/medium/f1_accel_ACCELERATING"]
)
delta_medium_d2c = (
    cand_full["proxy/medium/dynamic_to_constant_rate"]
    - ref_full["proxy/medium/dynamic_to_constant_rate"]
)

gate_checks = {
    "holdout_accel_nonnegative": delta_hold_accel >= 0.0,
    "full_accel_at_least_plus_0p002": delta_full_accel >= 0.002,
    "medium_accelerating_nonnegative": delta_medium_acc >= 0.0,
    "medium_dynamic_to_constant_nonpositive": delta_medium_d2c <= 0.0,
    "worst_rule_accel_not_below_minus_0p005": (
        worst_rule_accel_delta >= -0.005
    ),
}
LOCAL_ACCEPT = all(gate_checks.values())

print("rule accel deltas:", rule_accel_deltas)
print("delta holdout accel:", delta_hold_accel)
print("delta full accel   :", delta_full_accel)
print("delta full stage3  :", delta_full_stage3)
print("delta medium ACC   :", delta_medium_acc)
print("delta medium D->C  :", delta_medium_d2c)
print("gate checks        :", gate_checks)
print("LOCAL ACCEPT       :", LOCAL_ACCEPT)


,variant,split,stage3,accel,steer,medium_accelerating,medium_decelerating,medium_constant,medium_stopped,medium_dynamic_to_constant
0,v5-C,tune,0.712631,0.731080,0.669583,0.565702,0.629640,0.762142,0.987818,0.472169
1,v5-C,holdout,0.649522,0.689128,0.557108,0.578856,0.443299,0.866551,0.977625,0.419602
2,v5-C,full,0.702905,0.725738,0.649626,0.567345,0.602909,0.784105,0.986062,0.465787
3,v5-D best_accel_proxy.pt state=0,tune,0.717680,0.728775,0.691792,0.524095,0.667221,0.762324,0.988724,0.480847
4,v5-D best_accel_proxy.pt state=0,holdout,0.653020,0.695582,0.553706,0.595156,0.433996,0.868132,0.978504,0.439510
5,v5-D best_accel_proxy.pt state=0,full,0.707421,0.725305,0.665692,0.533394,0.636233,0.784643,0.986963,0.475827
6,v5-D best_accel_proxy.pt state=1,tune,0.718614,0.730109,0.691792,0.523213,0.665925,0.761913,0.988724,0.483598
7,v5-D best_accel_proxy.pt state=1,holdout,0.650024,0.691302,0.553706,0.595156,0.426471,0.867511,0.978504,0.448698
8,v5-D best_accel_proxy.pt state=1,full,0.707554,0.725495,0.665692,0.532638,0.634453,0.784212,0.986963,0.479360


rule accel deltas: {'sensitive': 0.00010150233933670094, 'medium': -0.0005386867010179763, 'conservative': -0.00029235286424122986}
delta holdout accel: 0.002173932679201407
delta full accel   : -0.00024317907530746474
delta full stage3  : 0.0046497347486362806
delta medium ACC   : -0.03470691177688945
delta medium D->C  : 0.013573819263666798
gate checks        : {'holdout_accel_nonnegative': True, 'full_accel_at_least_plus_0p002': False, 'medium_accelerating_nonnegative': False, 'medium_dynamic_to_constant_nonpositive': False, 'worst_rule_accel_not_below_minus_0p005': True}
LOCAL ACCEPT       : False


## 11. 최종 report 저장

`LOCAL_ACCEPT=False`면 v5-D를 submission 후보로 올리지 않고,
acceleration 설계의 결과/실패 모드를 기록한 뒤 v6-A backbone-only ablation으로 넘어간다.

`LOCAL_ACCEPT=True`여도 바로 DACON 제출을 의미하지 않는다.
그 다음 단계는 아래 optional public 50-label sanity이며, public labels로 weight를 다시 고르면 안 된다.

In [14]:
report = {
    "version": 1,
    "repo_commit": GIT_COMMIT,
    "v5d_training_commit": summary.get("git_commit"),
    "protocol": {
        "source_v5c_run": V5C_RUN_NAME,
        "segments": int(len(selected)),
        "tune_segments": int((selected["split"] == "tune").sum()),
        "holdout_segments": int((selected["split"] == "holdout").sum()),
        "clip_len": CLIP_LEN,
        "stride": OVERLAP_STRIDE,
        "center_floor": CENTER_FLOOR,
        "v5c_fusion": production_fusion.as_dict(),
        "proxy_rules": proxy_rules,
    },
    "checkpoint_fingerprints": fingerprint_df[
        ["checkpoint", "bytes", "sha256"]
    ].to_dict(orient="records"),
    "checkpoint_metadata": checkpoint_meta,
    "state_checkpoint": STATE_CKPT,
    "state_weight_candidates": STATE_WEIGHTS,
    "selected_state_weight": BEST_STATE_WEIGHT,
    "v5c_full": {
        "stage3": ref_full["proxy/robust_mean_stage3_score"],
        "accel": ref_full["proxy/robust_mean_accel_macro_f1"],
        "steer": ref_full["proxy/robust_mean_steer_macro_f1"],
        "medium_accelerating": ref_full[
            "proxy/medium/f1_accel_ACCELERATING"
        ],
        "medium_dynamic_to_constant": ref_full[
            "proxy/medium/dynamic_to_constant_rate"
        ],
    },
    "v5d_selected_full": {
        "stage3": cand_full["proxy/robust_mean_stage3_score"],
        "accel": cand_full["proxy/robust_mean_accel_macro_f1"],
        "steer": cand_full["proxy/robust_mean_steer_macro_f1"],
        "medium_accelerating": cand_full[
            "proxy/medium/f1_accel_ACCELERATING"
        ],
        "medium_dynamic_to_constant": cand_full[
            "proxy/medium/dynamic_to_constant_rate"
        ],
    },
    "deltas_vs_v5c": {
        "holdout_accel": delta_hold_accel,
        "full_accel": delta_full_accel,
        "full_stage3": delta_full_stage3,
        "medium_accelerating": delta_medium_acc,
        "medium_dynamic_to_constant": delta_medium_d2c,
        "full_rule_accel": rule_accel_deltas,
        "worst_rule_accel": worst_rule_accel_delta,
    },
    "gate_checks": gate_checks,
    "local_accept": LOCAL_ACCEPT,
    "note": (
        "Local proxy only. Released 50 labels are excluded from selection. "
        "Do not translate local delta directly to DACON leaderboard delta."
    ),
}

REPORT_PATH = REPORT_DIR / "v5d_overlap_accel_report.json"
REPORT_PATH.write_text(
    json.dumps(report, indent=2, default=str),
    encoding="utf-8",
)

gate_table.to_csv(
    REPORT_DIR / "v5d_vs_v5c_gate_table.csv",
    index=False,
)

print(json.dumps(report, indent=2, default=str))
print("saved:", REPORT_PATH)


{
  "version": 1,
  "repo_commit": "1ec2ba2e210905040a889ea5cd780d9da4aea462",
  "v5d_training_commit": "afaf7e9",
  "protocol": {
    "source_v5c_run": "vjepa21b_can_v5c_overlap_auxfusion",
    "segments": 25,
    "tune_segments": 20,
    "holdout_segments": 5,
    "clip_len": 32,
    "stride": 8,
    "center_floor": 0.25,
    "v5c_fusion": {
      "stop_weight": 0.75,
      "accel_weight": 0.0,
      "steer_weight": 0.3,
      "turn_weight": 0.3,
      "stop_temperature_mps": 0.25,
      "accel_temperature_mps2": 0.1,
      "steer_temperature_deg": 2.0
    },
    "proxy_rules": {
      "sensitive": {
        "stop_speed_mps": 0.3,
        "accel_deadzone_mps2": 0.1,
        "steer_deadzone_deg": 2.0
      },
      "medium": {
        "stop_speed_mps": 0.5,
        "accel_deadzone_mps2": 0.2,
        "steer_deadzone_deg": 5.0
      },
      "conservative": {
        "stop_speed_mps": 1.0,
        "accel_deadzone_mps2": 0.3,
        "steer_deadzone_deg": 8.0
      }
    }
  },
  "check

## 12. Optional — released 50 labels sanity check

**기본값은 실행하지 않는다.**

위 local selection을 끝낸 뒤 `RUN_PUBLIC_SANITY=True`로 바꿔서,
선택된 checkpoint와 `BEST_STATE_WEIGHT`를 그대로 사용한다.

여기서는:
- checkpoint를 바꾸지 않는다.
- state weight를 바꾸지 않는다.
- calibration threshold를 바꾸지 않는다.

즉 public 50 labels는 catastrophic mismatch를 보는 sanity check일 뿐이다.

In [15]:
RUN_PUBLIC_SANITY = False

if not RUN_PUBLIC_SANITY:
    print("PUBLIC SANITY SKIPPED (default)")
else:
    from blackbox_detection.stage3.dacon_inference import infer_public_frame_mapping

    # Load the already-selected checkpoint exactly as fixed above.
    selected_local = Path(
        fingerprint_df.loc[
            fingerprint_df["checkpoint"] == STATE_CKPT,
            "local_path",
        ].iloc[0]
    )
    load_checkpoint(
        selected_local,
        model=model,
        optimizer=None,
        scheduler=None,
        map_location="cpu",
        strict=True,
        restore_rng_state=False,
    )
    model.to(device).eval()
    model.requires_grad_(False)

    baseline_candidates = [
        DRIVE_ROOT / "Baseline.zip",
        Path("/content/Baseline.zip"),
    ]
    baseline_zip = next(
        (p for p in baseline_candidates if p.is_file()),
        None,
    )
    if baseline_zip is None:
        raise FileNotFoundError("Baseline.zip not found")

    public_root = Path("/content/v5d_public_stage3")
    if public_root.exists():
        shutil.rmtree(public_root)
    public_root.mkdir(parents=True)

    with zipfile.ZipFile(baseline_zip) as zf:
        members = [
            name for name in zf.namelist()
            if "/stage3/" in name.lower()
            or name.lower().startswith("data/stage3/")
            or name.lower().startswith("stage3/")
        ]
        if not members:
            raise RuntimeError("No Stage3 members in Baseline.zip")
        for name in members:
            zf.extract(name, public_root)

    label_files = list(public_root.rglob("labels.csv"))
    if not label_files:
        raise FileNotFoundError("labels.csv not found")
    labels_path = label_files[0]
    labels = pd.read_csv(labels_path)

    videos = {
        p.stem: p
        for p in [
            *public_root.rglob("*.mp4"),
            *public_root.rglob("*.MP4"),
        ]
    }

    public_parts = []
    for video_id, subset in labels.groupby("ID", sort=True):
        video_id = str(video_id)
        if video_id not in videos:
            raise FileNotFoundError(video_id)

        raw_stride, raw_offset = infer_public_frame_mapping(subset)
        feat = extract_video_features_v5d_fixed(
            model,
            videos[video_id],
            raw_stride=raw_stride,
            raw_offset=raw_offset,
            batch_size=int(v5c_cfg["data"]["batch_size"]),
        )
        feat.insert(0, "ID", video_id)
        public_parts.append(feat)

    public_features = pd.concat(public_parts, ignore_index=True)

    calibration_path = (
        REPO / v5c_cfg["public_sanity"]["calibration"]
    )
    calibration = json.loads(
        calibration_path.read_text(encoding="utf-8")
    )

    def apply_public_fixed(group: pd.DataFrame):
        accel_cfg = dict(calibration["accel"])
        steer_cfg = dict(calibration["steer"])
        g = group.sort_values("sample_index").reset_index(drop=True)
        g = smooth_feature_table(
            g,
            accel_window=int(accel_cfg["smoothing_window"]),
            steer_window=int(steer_cfg["smoothing_window"]),
        )

        rule = DecisionRule(
            stop_speed_mps=float(accel_cfg["stop_speed_mps"]),
            accel_pos_mps2=float(
                accel_cfg["accel_deadzone_pos_mps2"]
            ),
            accel_neg_mps2=float(
                accel_cfg["accel_deadzone_neg_mps2"]
            ),
            accel_bias_mps2=float(accel_cfg["accel_bias_mps2"]),
            steer_left_deg=float(steer_cfg["left_deadzone_deg"]),
            steer_right_deg=float(steer_cfg["right_deadzone_deg"]),
            steering_sign=int(steer_cfg["steering_sign"]),
            steering_bias_deg=float(steer_cfg["steering_bias_deg"]),
        )

        pred_a, pred_s = labels_with_state_fusion(
            g,
            rule=rule,
            base_fusion=production_fusion,
            state_weight=BEST_STATE_WEIGHT,
        )
        return pd.DataFrame({
            "ID": g["ID"].astype(str),
            "sample_index": g["sample_index"].astype(int),
            "pred_accel": pred_a,
            "pred_steer": pred_s,
        })

    pred = pd.concat(
        [
            apply_public_fixed(g)
            for _, g in public_features.groupby("ID", sort=False)
        ],
        ignore_index=True,
    )

    truth = labels.copy()
    truth["ID"] = truth["ID"].astype(str)
    merged = truth.merge(
        pred,
        on=["ID", "sample_index"],
        how="left",
        validate="many_to_one",
    )
    if merged[["pred_accel", "pred_steer"]].isna().any().any():
        raise RuntimeError("public prediction merge has missing rows")

    public_metrics = dacon_stage3_metrics(
        merged["accel_label"].to_numpy(),
        merged["pred_accel"].to_numpy(),
        merged["steer_label"].to_numpy(),
        merged["pred_steer"].to_numpy(),
    )

    public_report = {
        "checkpoint": STATE_CKPT,
        "state_weight": BEST_STATE_WEIGHT,
        "rows": int(len(merged)),
        "metrics": public_metrics,
        "warning": (
            "50 released labels only; no selection or retuning allowed."
        ),
    }
    (REPORT_DIR / "public_sanity_fixed.json").write_text(
        json.dumps(public_report, indent=2),
        encoding="utf-8",
    )
    print(json.dumps(public_report, indent=2))


PUBLIC SANITY SKIPPED (default)


## 해석 기준

이 notebook의 핵심 질문은 하나다.

> **v5-C와 완전히 같은 complete-segment overlap inference에서,
> v5-D가 acceleration을 실제로 개선했는가?**

특히 training diagnostic에서 보였던
`ACCELERATING` 하락과 `dynamic→CONSTANT` 악화가 overlap에서도 남는다면,
v5-D에 추가 시간을 쓰지 않고 현재 acceleration 설계를 기록한 뒤
**v6-A: 동일 head/loss/data/split/inference + VideoMamba backbone-only ablation**
으로 넘어가는 것이 목적이다.

반대로 strict local gate를 통과하면 public 50-label sanity만 추가하고,
그 결과를 leaderboard delta로 과해석하지 않는다.